In [34]:
import ExTRA as ex
import numpy as np
import astroquery
import astropy
import matplotlib.pyplot as plt

### This notebook shows how to
## 1) create a consistent model between gaia and hip
## 2) apply light time delay correction to timestamps
## 3) apply secular acceleration correction to individual measurements

# Correction using functions, using Nu Octantis , aka HIP107089 as an example

In [35]:


#Nu Oct as example
#HIP107089
#inputs:
gaia_asc=(21 +41/60 +28.5420355132/3600 )*15 #deg
gaia_dec=(-77 -23/60 -24.031541832/3600) #deg
gaia_sss=np.array([gaia_asc,gaia_dec,51.5172,68.656,-250.044]) # in radians for propagator
v_rad_gaia=34.4 #km/s


hip_sss=np.array([np.degrees(5.6787539579),np.degrees(-1.3507009230),47.16,66.40,-239.10])
hip_iad,t_hip=ex.hip_read("data/nu_oct/HIP107089_esa.d")



## 1) consistent

In [36]:

#hip iad as per usual
#models both in deg deg mas mas/yr mas/yr
#v_rad at gaia epoch in km/s
#consistent models:
abs_consistent_new,hip_sss_new,v_hip=ex.consistent_models(hip_iad,hip_sss,gaia_sss,v_rad_gaia)



In [37]:
hip_sss_new

array([ 325.37121848,  -77.3918319 ,   51.51474908,   68.65923348,
       -250.01752725])

## 2) ltd, correcting timestamps

In [38]:
t_hip_r=ex.hip_JD(hip_iad,format="relative") 
#the relative format is important, it gives timestamps as years relative to 1991.25
#for gaia data, use function called "gaia_JD" the same way
tau=ex.ltd_accurate(t_hip_r,hip_sss_new,v_hip)
t_hip_ltd=t_hip_r+tau

## 3) secular, can be done with ltd beforehand or without

In [39]:
hip_sss_new_rad=np.concatenate([np.radians(hip_sss_new[:2]),hip_sss_new[2:]])
hip_sss_new_rad

array([   5.67879905,   -1.35074228,   51.51474908,   68.65923348,
       -250.01752725])

In [40]:
testing_prop=ex.EpochPropagation()
diff=testing_prop.propagate_astrometry(*hip_sss_new_rad,v_hip,1991.25,t_hip_ltd[100]+1991.25)[:-1]-hip_sss_new_rad



In [41]:
np.degrees(diff[:2]) *3.6e6  # ex.sss_model(hip_sss_new,t_hip[100],Sepoch=ex.J1991())

array([ 229.36348857, -182.31091149])

In [42]:
shift_asc,shift_dec=ex.secular_shift(t_hip_ltd,hip_sss_new,v_hip)

#please see that we used the already ltd corrected timestamps

#removing the shift accordingly:
abs_final=abs_consistent_new-(shift_asc*hip_iad[0]+shift_dec*hip_iad[1])



hip_iad_final=hip_iad.copy()
#final corrected residual data:
hip_iad_final[-2]=abs_final

#if you want to rewrite this to a file, you also need to change the timestamps!!!!!
#this is NOT easy since multiple columns in the HIP data format must be edited.




In [51]:
ex.secular_zech(np.array([0,0,550,-800,10300]))*1000

np.float64(4.459848763242965)

In [ ]:
ex.secular_acceleration(hip_sss_new,v_hip)[1:3] #mas

array([-0.00024888,  0.00090626])

## 3.1) change of standard model parameters

In [45]:
secular_change=ex.secular_acceleration(hip_sss_new,v_hip)
print("changes of par,mu_a and mu_d per year:",secular_change)
print("this is just a fun fact and not important for computing")

changes of par,mu_a and mu_d per year: [-9.33651892e-05 -2.48875611e-04  9.06262154e-04  2.99903595e-05]
this is just a fun fact and not important for computing


## Outcome

In [46]:
print("old likelihood:",ex.loglikelihood(hip_iad[-2],hip_iad[-1],0))
print("new likelihood:",ex.loglikelihood(abs_consistent_new,hip_iad[-1],0))
print("new likelihood:",ex.loglikelihood(abs_final,hip_iad[-1],0))

old likelihood: 530.8247862687826
new likelihood: 555.4917014915138
new likelihood: 555.4920709867171


the likelihood got worse as expected, since hipparcos on its own used to have the optimal fit with the data available. the now new residuals are consistent with gaias single steller solution and can therefor easily be fit in combination. the single stellar solution fit corrections can now be applied to both of the residuals in the same way

## Bonus:

## Here i compare to a Lindegren paper to see if our estimates are comparable, and they are:
#### (im using more accurate proper motions than he did)

In [47]:
#L. Lindegren 2021:
#The largest changes are expected for Barnards’s star
#(HIP 87937) owing to its sizeable parallax ('547 mas), proper
#motion ('10 393 mas yr−1), and radial velocity ('−110 km s−1).
#For this star, the perspective effects produce, over the 24.75 yr,
#a position difference of about 393 mas, an increase in the paral-
#lax by 0.84 mas, and an increase in the proper motion by about
#32 mas yr−1.

barnard=np.array([0,0,547,-801.551,10362.394])
v_barnard=-110
sec_lindegren=np.array(ex.secular_acceleration(barnard,v_barnard))* 24.75
shift_lindegren=ex.secular_shift(24.75*np.ones([5]),barnard,v_barnard) ####??????
print("change in parallax and proper motion over 24.75 years:",sec_lindegren)
print("total shift over 24.75 years:",shift_lindegren)

change in parallax and proper motion over 24.75 years: [ 0.83309775 -2.44157344 31.56448685  0.11232985]
total shift over 24.75 years: [[ 3.59912845e+02  3.59912845e+02  3.59912845e+02  3.59912845e+02
   3.59912845e+02]
 [-5.50381330e-03 -5.50381330e-03 -5.50381330e-03 -5.50381330e-03
  -5.50381330e-03]]
